In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier
from sklearn.metrics import classification_report,accuracy_score
from sklearn.utils.multiclass import unique_labels

In [15]:
# 📥 2. Load Data
df = pd.read_csv("wishlist_flat.csv")  # Make sure this is pre-exploded per kategori


In [16]:
df.head()

,user_id,gender,age,item_id,kategori
0,3431822,Male,2024.0,292801,poem & short story
1,3431822,Male,2024.0,292746,poem & short story
2,6,Male,47.0,303475,national
3,3431823,Male,2024.0,291111,children age 4-7
4,917,Female,NaN,193309,parenting & relationships


In [17]:
# 🧼 3. Clean & Normalize
df['kategori'] = df['kategori'].str.strip().str.lower()
df['gender'] = df['gender'].str.title().fillna('Unknown')

In [18]:
# 🧽 4. Remove rare/irrelevant categories (including "luxury")
min_category_support = 10
valid_categories = df['kategori'].value_counts()
valid_categories = valid_categories[valid_categories >= min_category_support].index
df = df[df['kategori'].isin(valid_categories)]

In [19]:
# 🧼 5. Remove users with < 3 wishlist entries
wishlist_counts = df.groupby('user_id').size()
valid_users = wishlist_counts[wishlist_counts >= 3].index
df = df[df['user_id'].isin(valid_users)]

In [20]:
user_cat_counts = (
    df.groupby(['user_id', 'kategori'])
    .size()
    .reset_index(name='count')
)
top_cat = (
    user_cat_counts
    .sort_values(['user_id', 'count'], ascending=[True, False])
    .drop_duplicates('user_id')
    .rename(columns={'kategori': 'target_category'})
)

In [21]:
# 👤 7. Get user features
user_features = df.groupby('user_id')[['gender', 'age']].first().reset_index()
user_features['gender'] = user_features['gender'].map({'Male': 0, 'Female': 1, 'Unknown': -1})
user_features = user_features.dropna(subset=['age'])

In [22]:
# 🔗 8. Merge features with target labels
train_df = user_features.merge(top_cat[['user_id', 'target_category']], on='user_id')

In [23]:
# 🎯 9. Encode labels
label_encoder = LabelEncoder()
train_df['target_encoded'] = label_encoder.fit_transform(train_df['target_category'])

In [24]:

# ⚠️ 10. Final filtering for categories with enough users
valid_cats = train_df['target_category'].value_counts()
valid_cats = valid_cats[valid_cats >= 2].index
train_df_filtered = train_df[train_df['target_category'].isin(valid_cats)].copy()

In [25]:
# 🧪 12. Train/Test Split
X = train_df_filtered[['gender', 'age']]
y = train_df_filtered['target_encoded']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# 🌳 13. Train Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
# clf = HistGradientBoostingClassifier()
clf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [26]:
 # 📊 14. Evaluation
y_pred = clf.predict(X_test)
actual_labels = unique_labels(y_test, y_pred)
target_names = label_encoder.inverse_transform(actual_labels)
print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=target_names))

📊 Classification Report:

                                       precision    recall  f1-score   support

                        adult fiction       0.00      0.00      0.00       118
             art, music & photography       0.00      0.00      0.00         6
             automotive & motorcycles       0.00      0.00      0.00         1
                biographies & memoirs       0.00      0.00      0.00        10
                 business & investing       0.23      0.08      0.12       166
                             children       0.00      0.00      0.00         5
                     children age 0-3       0.00      0.00      0.00         1
                     children age 4-7       0.00      0.00      0.00        16
                    children age 8-12       0.00      0.00      0.00        19
                         christianity       0.00      0.00      0.00        10
                             classics       0.00      0.00      0.00        25
              comics & gr

D:\belajar\the_projects\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\belajar\the_projects\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\belajar\the_projects\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [27]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.23


In [33]:
# 🔮 15. Predict top-N categories per user
def predict_top_categories(gender, age, top_n=3):
    g = {'Male': 0, 'Female': 1, 'Unknown': -1}.get(gender.title(), -1)
    probs = clf.predict_proba([[g, age]])[0]
    top_indices = probs.argsort()[::-1][:top_n]
    return label_encoder.inverse_transform(top_indices)

In [34]:
# 📦 16. Output: Age + Predicted Categories
unique_ages = sorted(train_df_filtered['age'].unique())
output = []
for age in sorted(train_df_filtered['age'].unique()):
    for gender in ['Male', 'Female', 'Unknown']:
        top_preds = predict_top_categories(gender, age, top_n=3)
        output.append({
        'age': int(age),
        'gender': gender,
        'predicted_top_categories': ', '.join(top_preds)
    })

D:\belajar\the_projects\venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
D:\belajar\the_projects\venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
D:\belajar\the_projects\venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
D:\belajar\the_projects\venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
D:\belajar\the_projects\venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with featur

In [35]:
output

[{'age': -7973,
  'gender': 'Male',
  'predicted_top_categories': 'adult fiction, motivation & self-help, business & investing'},
 {'age': -7973,
  'gender': 'Female',
  'predicted_top_categories': 'motivation & self-help, fiction & literature, technology, games & gadget'},
 {'age': -7973,
  'gender': 'Unknown',
  'predicted_top_categories': 'adult fiction, motivation & self-help, business & investing'},
 {'age': 0,
  'gender': 'Male',
  'predicted_top_categories': 'motivation & self-help, business & investing, technology, games & gadget'},
 {'age': 0,
  'gender': 'Female',
  'predicted_top_categories': 'motivation & self-help, fiction & literature, technology, games & gadget'},
 {'age': 0,
  'gender': 'Unknown',
  'predicted_top_categories': 'motivation & self-help, business & investing, technology, games & gadget'},
 {'age': 1,
  'gender': 'Male',
  'predicted_top_categories': 'education & test preparation, motivation & self-help, history'},
 {'age': 1,
  'gender': 'Female',
  'predi